In [1]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import root_mean_squared_error

DATA_ROOT = "data"
MODELS = ["Model_1", "Model_2"]

SPLIT_TRAIN = "train"
SPLIT_TEST  = "test"

T = 445
WARMUP = 9

SEED = 6
rng = np.random.RandomState(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
USE_AMP = (device.type == "cuda")
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
USE_SCALER = USE_AMP and (AMP_DTYPE == torch.float16)
print("amp:", USE_AMP, "dtype:", AMP_DTYPE, "scaler:", USE_SCALER)

device: cuda
amp: True dtype: torch.bfloat16 scaler: False


In [2]:
if os.path.exists("sample_submission.parquet"):
    sub = pd.read_parquet("early_data/sample_submission.parquet")
else:
    sub = pd.read_csv("early_data/sample_submission.csv")

print("sample submission:", sub.shape)
print("columns:", list(sub.columns))

sample submission: (50910192, 6)
columns: ['row_id', 'model_id', 'event_id', 'node_type', 'node_id', 'water_level']


In [3]:
COL_ROW   = "row_id"
COL_MODEL = "model_id"
COL_EVENT = "event_id"
COL_TYPE  = "node_type"
COL_NODE  = "node_id"
COL_Y     = "water_level"

In [4]:
def list_events(model_name, split):
    base = f"{DATA_ROOT}/{model_name}/{split}"
    event_dirs = sorted([
        d for d in os.listdir(base)
        if d.startswith("event_") and os.path.isdir(os.path.join(base, d))
    ])
    return base, event_dirs

def load_static(model_name):
    train_dir = f"{DATA_ROOT}/{model_name}/train"

    df_1d_edge_index      = pd.read_csv(f"{train_dir}/1d_edge_index.csv")
    df_1d_edges_static    = pd.read_csv(f"{train_dir}/1d_edges_static.csv")
    df_1d_nodes_static    = pd.read_csv(f"{train_dir}/1d_nodes_static.csv")
    df_1d2d_connections   = pd.read_csv(f"{train_dir}/1d2d_connections.csv")

    df_2d_edge_index      = pd.read_csv(f"{train_dir}/2d_edge_index.csv")
    df_2d_edges_static    = pd.read_csv(f"{train_dir}/2d_edges_static.csv")
    df_2d_nodes_static    = pd.read_csv(f"{train_dir}/2d_nodes_static.csv")

    # your cleanup
    df_2d_nodes_static["area"] = df_2d_nodes_static["area"].clip(lower=0)
    mask_nan = df_2d_nodes_static["min_elevation"].isna()
    df_2d_nodes_static.loc[mask_nan, "min_elevation"] = df_2d_nodes_static.loc[mask_nan, "elevation"]
    mask = df_2d_nodes_static["min_elevation"] > df_2d_nodes_static["elevation"]
    df_2d_nodes_static.loc[mask, "min_elevation"] = df_2d_nodes_static.loc[mask, "elevation"]

    return (df_1d_edge_index, df_1d_edges_static, df_1d_nodes_static, df_1d2d_connections,
            df_2d_edge_index, df_2d_edges_static, df_2d_nodes_static)

def split_events(event_dirs, val_frac=0.2, seed=7):
    r = np.random.RandomState(seed)
    n_val = max(1, int(val_frac * len(event_dirs)))
    val = sorted(r.choice(event_dirs, size=n_val, replace=False).tolist())
    tr = [e for e in event_dirs if e not in set(val)]
    return tr, val

def save_ckpt(path, model, opt, step):
    torch.save({
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "step": step
    }, path)

def load_ckpt(path, model, opt):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["opt"])
    return ckpt["step"]

# 3) Cache dynamic per event to memmap for fast training/inference
#    We only cache 2D nodes dynamic: water_level + rainfall

In [5]:
def ensure_cache_dir(model_name, split):
    cache_dir = f"{DATA_ROOT}/{model_name}/{split}/_cache"
    os.makedirs(cache_dir, exist_ok=True)
    return cache_dir

def cache_event_2d(model_name, split, ev, N2, id2i_2d):
    base = f"{DATA_ROOT}/{model_name}/{split}"
    cache_dir = ensure_cache_dir(model_name, split)
    out_dir = f"{cache_dir}/{ev}"
    os.makedirs(out_dir, exist_ok=True)

    out_path = f"{out_dir}/2d_dyn.dat"
    if os.path.exists(out_path):
        return

    mm = np.memmap(out_path, mode="w+", dtype=np.float32, shape=(T, N2, 2))
    mm[:] = 0.0

    usecols = ["timestep", "node_idx", "water_level", "rainfall"]
    f = f"{base}/{ev}/2d_nodes_dynamic_all.csv"

    for chunk in pd.read_csv(f, usecols=usecols, chunksize=2_000_000):
        t = chunk["timestep"].to_numpy(np.int64)
        nid = chunk["node_idx"].to_numpy(np.int64)
        idx = np.fromiter((id2i_2d[int(x)] for x in nid), dtype=np.int64, count=nid.shape[0])

        mm[t, idx, 0] = chunk["water_level"].to_numpy(np.float32)
        mm[t, idx, 1] = chunk["rainfall"].to_numpy(np.float32)

    mm.flush()
    print("[cache] 2D", model_name, split, ev)

def cache_event_1d(model_name, split, ev, N1, id2i_1d):
    base = f"{DATA_ROOT}/{model_name}/{split}"
    cache_dir = ensure_cache_dir(model_name, split)
    out_dir = f"{cache_dir}/{ev}"
    os.makedirs(out_dir, exist_ok=True)

    out_path = f"{out_dir}/1d_dyn.dat"
    if os.path.exists(out_path):
        return

    mm = np.memmap(out_path, mode="w+", dtype=np.float32, shape=(T, N1, 1))
    mm[:] = 0.0

    usecols = ["timestep", "node_idx", "water_level"]
    f = f"{base}/{ev}/1d_nodes_dynamic_all.csv"

    for chunk in pd.read_csv(f, usecols=usecols, chunksize=2_000_000):
        t = chunk["timestep"].to_numpy(np.int64)
        nid = chunk["node_idx"].to_numpy(np.int64)
        idx = np.fromiter((id2i_1d[int(x)] for x in nid), dtype=np.int64, count=nid.shape[0])
        mm[t, idx, 0] = chunk["water_level"].to_numpy(np.float32)

    mm.flush()
    print("[cache] 1D", model_name, split, ev)

def open_mm_2d(model_name, split, ev, N2):
    cache_dir = ensure_cache_dir(model_name, split)
    path = f"{cache_dir}/{ev}/2d_dyn.dat"
    return np.memmap(path, mode="r", dtype=np.float32, shape=(T, N2, 2))

def open_mm_1d(model_name, split, ev, N1):
    cache_dir = ensure_cache_dir(model_name, split)
    path = f"{cache_dir}/{ev}/1d_dyn.dat"
    return np.memmap(path, mode="r", dtype=np.float32, shape=(T, N1, 1))

# 4) Build graph packs

In [6]:
def build_pack(df_1d_edge_index, df_1d_edges_static, df_1d_nodes_static,
               df_2d_edge_index, df_2d_edges_static, df_2d_nodes_static):

    node_ids_1d = df_1d_nodes_static["node_idx"].to_numpy(np.int64)
    node_ids_2d = df_2d_nodes_static["node_idx"].to_numpy(np.int64)

    N1 = node_ids_1d.shape[0]
    N2 = node_ids_2d.shape[0]

    id2i_1d = {int(n): i for i, n in enumerate(node_ids_1d)}
    id2i_2d = {int(n): i for i, n in enumerate(node_ids_2d)}

    # 2D edges
    ei2 = df_2d_edge_index.to_numpy()
    src2 = ei2[:, -2].astype(np.int64)
    dst2 = ei2[:, -1].astype(np.int64)
    src2 = np.array([id2i_2d[int(x)] for x in src2], dtype=np.int64)
    dst2 = np.array([id2i_2d[int(x)] for x in dst2], dtype=np.int64)

    
    node_num = df_2d_nodes_static.select_dtypes(include=[np.number]).copy()
    for c in ["node_idx"]:
        if c in node_num.columns:
            node_num = node_num.drop(columns=[c])

    edge_num = df_2d_edges_static.select_dtypes(include=[np.number]).copy()
    for c in ["edge_idx"]:
        if c in edge_num.columns:
            edge_num = edge_num.drop(columns=[c])

    node_np = node_num.to_numpy(np.float32)
    node_mu = node_np.mean(axis=0, keepdims=True)
    node_sd = node_np.std(axis=0, keepdims=True)
    node_sd[node_sd < 1e-6] = 1e-6
    node_np = (node_np - node_mu) / node_sd

    edge_np = edge_num.to_numpy(np.float32)
    edge_mu = edge_np.mean(axis=0, keepdims=True)
    edge_sd = edge_np.std(axis=0, keepdims=True)
    edge_sd[edge_sd < 1e-6] = 1e-6
    edge_np = (edge_np - edge_mu) / edge_sd

    node_static_2d = torch.tensor(node_np, device=device)
    edge_attr_2d = torch.tensor(edge_np, device=device)

    # CSR for 2D neighbor sampling
    order = np.argsort(src2)
    src2_s = src2[order]
    dst2_s = dst2[order]
    indptr2 = np.zeros(N2 + 1, dtype=np.int64)
    np.add.at(indptr2, src2_s + 1, 1)
    indptr2 = np.cumsum(indptr2)

    pack = {
        "node_ids_1d": node_ids_1d,
        "node_ids_2d": node_ids_2d,
        "id2i_1d": id2i_1d,
        "id2i_2d": id2i_2d,
        "N1": N1,
        "N2": N2,
        "src2": src2,
        "dst2": dst2,
        "src2_s": src2_s,
        "dst2_s": dst2_s,
        "indptr2": indptr2,
        "node_static_2d": node_static_2d,
        "edge_attr_2d": edge_attr_2d,
    }
    return pack

def sample_subgraph_2d(pack, seed_n=8192, neigh_k=8):
    N2 = pack["N2"]
    src2 = pack["src2"]
    dst2 = pack["dst2"]
    src2_s = pack["src2_s"]
    dst2_s = pack["dst2_s"]
    indptr2 = pack["indptr2"]

    seeds = rng.randint(0, N2, size=seed_n, dtype=np.int64)
    nodes = set(seeds.tolist())

    for u in seeds:
        a, b = indptr2[u], indptr2[u+1]
        if b <= a:
            continue
        nbrs = dst2_s[a:b]
        if nbrs.shape[0] > neigh_k:
            take = rng.choice(nbrs.shape[0], size=neigh_k, replace=False)
            nbrs = nbrs[take]
        for v in nbrs.tolist():
            nodes.add(int(v))

    nodes = np.fromiter(nodes, dtype=np.int64)
    nodes.sort()

    mask = np.zeros(N2, dtype=np.bool_)
    mask[nodes] = True
    e_mask = mask[src2] & mask[dst2]
    e_idx = np.where(e_mask)[0].astype(np.int64)

    re = np.full(N2, -1, dtype=np.int64)
    re[nodes] = np.arange(nodes.shape[0], dtype=np.int64)

    src_l = re[src2[e_idx]]
    dst_l = re[dst2[e_idx]]
    return nodes, e_idx, src_l, dst_l

# 5) MPNN + GRU model

In [7]:
class MPNN_GRU_Strong(nn.Module):
    def __init__(self, x_dim, s_dim, e_dim, h=128, s_emb=64, e_emb=64):
        super().__init__()
        self.h = h
        self.s_proj = nn.Linear(s_dim, s_emb)
        self.e_proj = nn.Linear(e_dim, e_emb)
        self.x_proj = nn.Linear(x_dim + s_emb, h)

        self.msg_mlp = nn.Sequential(
            nn.Linear(h + e_emb, h),
            nn.ReLU(),
            nn.Linear(h, h),
        )
        self.edge_gate = nn.Sequential(
            nn.Linear(e_emb, h),
            nn.Sigmoid()
        )

        self.gru = nn.GRUCell(h + h, h)
        self.norm = nn.LayerNorm(h)

        self.readout = nn.Sequential(
            nn.Linear(h, h),
            nn.ReLU(),
            nn.Linear(h, 1)
        )

    def step(self, x_t, s_emb, src, dst, e_emb, h, rounds=2):
        xh = self.x_proj(torch.cat([x_t, s_emb], dim=1))
        N = xh.shape[0]
        for _ in range(rounds):
            m_in = torch.cat([h[src], e_emb], dim=1)
            m = self.msg_mlp(m_in) * self.edge_gate(e_emb)

            agg = torch.zeros(N, self.h, device=h.device, dtype=m.dtype)
            agg.index_add_(0, dst, m)
            deg = torch.zeros(N, device=h.device, dtype=agg.dtype)
            deg.index_add_(0, dst, torch.ones_like(dst, dtype=agg.dtype))
            agg = agg / deg.clamp_min(1.0).unsqueeze(1)

            h = self.gru(torch.cat([xh, agg], dim=1), h)
            h = self.norm(h)

        delta = self.readout(h).squeeze(1)
        delta = 0.5 * torch.tanh(delta)
        return h, delta

def ss_prob(step, warm=2000, maxp=0.6):
    p = min(1.0, step / warm)
    return float(maxp * p)

In [8]:
def train_system(model_name, pack, event_dirs, steps=20000, L=32, seed_n=8192, neigh_k=8, rounds=2):
    node_static_2d = pack["node_static_2d"]
    edge_attr_2d = pack["edge_attr_2d"]
    N2 = pack["N2"]

    model = MPNN_GRU_Strong(
        x_dim=4,  # [wl_in, rain, rain_sum3, rain_sum10]
        s_dim=node_static_2d.shape[1],
        e_dim=edge_attr_2d.shape[1],
        h=128,
        s_emb=64,
        e_emb=64
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler(device.type, enabled=USE_SCALER)
    start_step = 0
    ckpt_path = f"{model_name}_2d_ckpt.pt"
    if os.path.exists(ckpt_path):
        start_step = load_ckpt(ckpt_path, model, opt)
        print("resume", model_name, "from step", start_step)

    model.train()
    global_step = start_step

    for it in range(steps):
        ev = event_dirs[rng.randint(0, len(event_dirs))]
        mm = open_mm_2d(model_name, "train", ev, N2)

        t0 = rng.randint(0, T - (L + 1))
        nodes, e_idx, src_l, dst_l = sample_subgraph_2d(pack, seed_n=seed_n, neigh_k=neigh_k)

        X = mm[t0:t0+L+1, nodes, :]  # (L+1, n, 2)
        wl = torch.tensor(X[:, :, 0], device=device)
        rain = torch.tensor(X[:, :, 1], device=device)


        start = max(0, t0 - 9)
        rbuf = torch.tensor(mm[start:t0+L+1, nodes, 1], device=device)
        cs = torch.cumsum(rbuf, dim=0)
        cs = torch.cat([torch.zeros(1, cs.shape[1], device=device, dtype=cs.dtype), cs], dim=0)

        i0 = t0 - start
        idx = torch.arange(i0, i0 + (L + 1), device=device)

        rain_sum3 = cs[idx + 1] - cs[torch.clamp(idx + 1 - 3, min=0)]
        rain_sum10 = cs[idx + 1] - cs[torch.clamp(idx + 1 - 10, min=0)]

        s = node_static_2d[nodes]
        e = edge_attr_2d[e_idx]

        src = torch.tensor(src_l, dtype=torch.long, device=device)
        dst = torch.tensor(dst_l, dtype=torch.long, device=device)

        s_emb = model.s_proj(s)
        e_emb = model.e_proj(e)

        h = torch.zeros(nodes.shape[0], model.h, device=device)
        p_ss = ss_prob(global_step)

        opt.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = 0.0
            n_loss = 0
            wl_in = wl[0]

            for t in range(0, L):
                x_t = torch.stack([wl_in, rain[t], rain_sum3[t], rain_sum10[t]], dim=1)
                h, d = model.step(x_t, s_emb, src, dst, e_emb, h, rounds=rounds)

                wl_pred_next = wl_in + d
                wl_true_next = wl[t+1]

                if (t0 + t) >= WARMUP:
                    loss = loss + F.mse_loss(wl_pred_next, wl_true_next)
                    n_loss += 1

                if (t0 + t + 1) >= WARMUP:
                    wl_in = p_ss * wl_pred_next + (1.0 - p_ss) * wl_true_next
                else:
                    wl_in = wl_true_next

            loss = loss / max(1, n_loss)


        if USE_SCALER:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        global_step += 1

        if (it + 1) % 100 == 0:
            print(f"[{model_name}] it={it+1} loss={float(loss.item()):.6f} ss={p_ss:.3f} ev={ev} t0={t0} n={nodes.shape[0]} e={len(e_idx)}")
        
        if (it + 1) % 500 == 0:
            save_ckpt(ckpt_path, model, opt, global_step)

    return model

In [9]:
@torch.no_grad()
def predict_event_2d_full(model_name, pack, model, ev, split="test", rounds=2):
    model.eval()

    N2 = pack["N2"]
    mm = open_mm_2d(model_name, split, ev, N2)

    wl_truth = torch.tensor(mm[:, :, 0], device=device)
    rain = torch.tensor(mm[:, :, 1], device=device)

    rain_sum3 = torch.zeros_like(rain)
    rain_sum10 = torch.zeros_like(rain)
    for t in range(T):
        a3 = max(0, t-2)
        a10 = max(0, t-9)
        rain_sum3[t] = rain[a3:t+1].sum(dim=0)
        rain_sum10[t] = rain[a10:t+1].sum(dim=0)

    node_static_2d = pack["node_static_2d"]
    edge_attr_2d = pack["edge_attr_2d"]

    src_full = torch.tensor(pack["src2"], dtype=torch.long, device=device)
    dst_full = torch.tensor(pack["dst2"], dtype=torch.long, device=device)

    with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        s_emb = model.s_proj(node_static_2d)
        e_emb = model.e_proj(edge_attr_2d)

    wl_pred = torch.zeros_like(wl_truth)
    wl_pred[:WARMUP+1] = wl_truth[:WARMUP+1]

    h = torch.zeros(N2, model.h, device=device)

    for t in range(WARMUP, T-1):
        wl_in = wl_pred[t]
        x_t = torch.stack([wl_in, rain[t], rain_sum3[t], rain_sum10[t]], dim=1)
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            h, d = model.step(x_t, s_emb, src_full, dst_full, e_emb, h, rounds=rounds)
        wl_pred[t+1] = wl_in + d

    return wl_pred.detach().cpu().numpy().astype(np.float32)

In [10]:
systems = {}

for model_name in MODELS:
    print("\nsetup", model_name)
    (df_1d_edge_index, df_1d_edges_static, df_1d_nodes_static, df_1d2d_connections,
     df_2d_edge_index, df_2d_edges_static, df_2d_nodes_static) = load_static(model_name)

    pack = build_pack(df_1d_edge_index, df_1d_edges_static, df_1d_nodes_static,
                      df_2d_edge_index, df_2d_edges_static, df_2d_nodes_static)

    train_base, train_events = list_events(model_name, "train")
    test_base, test_events = list_events(model_name, "test")

    # cache both train and test for 1D and 2D (so we can warmup from truth in test if it exists)
    for ev in train_events:
        cache_event_2d(model_name, "train", ev, pack["N2"], pack["id2i_2d"])
        cache_event_1d(model_name, "train", ev, pack["N1"], pack["id2i_1d"])

    for ev in test_events:
        cache_event_2d(model_name, "test", ev, pack["N2"], pack["id2i_2d"])
        cache_event_1d(model_name, "test", ev, pack["N1"], pack["id2i_1d"])

    train_events, val_events = split_events(train_events, val_frac=0.2, seed=SEED)

    systems[model_name] = {
        "pack": pack,
        "train_events": train_events,
        "val_events": val_events,
        "test_events": test_events,
    }


setup Model_1

setup Model_2


In [11]:
trained = {}
count = 0
train_steps = [5000, 20000]
for model_name in MODELS:
    print("\n=== train", model_name, "===")
    pack = systems[model_name]["pack"]
    train_events = systems[model_name]["train_events"]

    model = train_system(
        model_name=model_name,
        pack=pack,
        event_dirs=train_events,
        steps=train_steps[count],
        L=32,
        seed_n=8192,
        neigh_k=8,
        rounds=2
    )
    count+=1

    trained[model_name] = model
    torch.save(model.state_dict(), f"{model_name}_mpnn_gru.pt")
    print("saved", f"{model_name}_mpnn_gru.pt")


=== train Model_1 ===


C:\Users\rowes\AppData\Local\Temp\ipykernel_17676\1318601664.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=device)


resume Model_1 from step 15500
[Model_1] it=100 loss=0.000014 ss=0.600 ev=event_14 t0=271 n=3697 e=7849
[Model_1] it=200 loss=0.000001 ss=0.600 ev=event_78 t0=272 n=3693 e=7841
[Model_1] it=300 loss=0.000230 ss=0.600 ev=event_47 t0=73 n=3700 e=7864
[Model_1] it=400 loss=0.000001 ss=0.600 ev=event_49 t0=276 n=3694 e=7846
[Model_1] it=500 loss=0.001696 ss=0.600 ev=event_13 t0=167 n=3698 e=7864
[Model_1] it=600 loss=0.003009 ss=0.600 ev=event_41 t0=9 n=3686 e=7806
[Model_1] it=700 loss=5050.370117 ss=0.600 ev=event_41 t0=186 n=3694 e=7840
[Model_1] it=800 loss=0.000001 ss=0.600 ev=event_93 t0=226 n=3701 e=7877
[Model_1] it=900 loss=0.001282 ss=0.600 ev=event_11 t0=282 n=3695 e=7851
[Model_1] it=1000 loss=0.000000 ss=0.600 ev=event_57 t0=266 n=3702 e=7879
[Model_1] it=1100 loss=5050.708984 ss=0.600 ev=event_14 t0=204 n=3698 e=7858
[Model_1] it=1200 loss=0.000001 ss=0.600 ev=event_24 t0=394 n=3702 e=7876
[Model_1] it=1300 loss=0.001425 ss=0.600 ev=event_25 t0=272 n=3693 e=7837
[Model_1] it=


=== train Model_1 ===
[Model_1] it=100 loss=0.000093 ss=0.030 ev=event_14 t0=271 n=3697 e=7849
[Model_1] it=200 loss=0.000040 ss=0.060 ev=event_78 t0=272 n=3693 e=7841
[Model_1] it=300 loss=0.000970 ss=0.090 ev=event_47 t0=73 n=3700 e=7864
[Model_1] it=400 loss=0.000027 ss=0.120 ev=event_49 t0=276 n=3694 e=7846
[Model_1] it=500 loss=0.000708 ss=0.150 ev=event_13 t0=167 n=3698 e=7864
[Model_1] it=600 loss=0.001573 ss=0.180 ev=event_41 t0=9 n=3686 e=7806
[Model_1] it=700 loss=3394.995361 ss=0.210 ev=event_41 t0=186 n=3694 e=7840
[Model_1] it=800 loss=0.000012 ss=0.240 ev=event_93 t0=226 n=3701 e=7877
[Model_1] it=900 loss=0.001066 ss=0.270 ev=event_11 t0=282 n=3695 e=7851
[Model_1] it=1000 loss=0.000012 ss=0.300 ev=event_57 t0=266 n=3702 e=7879
[Model_1] it=1100 loss=3637.229736 ss=0.330 ev=event_14 t0=204 n=3698 e=7858
[Model_1] it=1200 loss=0.000013 ss=0.360 ev=event_24 t0=394 n=3702 e=7876
[Model_1] it=1300 loss=0.000252 ss=0.390 ev=event_25 t0=272 n=3693 e=7837
[Model_1] it=1400 loss=0.001020 ss=0.420 ev=event_34 t0=61 n=3695 e=7846
[Model_1] it=1500 loss=4063.272705 ss=0.450 ev=event_30 t0=199 n=3702 e=7881
[Model_1] it=1600 loss=0.000027 ss=0.480 ev=event_84 t0=220 n=3699 e=7868
[Model_1] it=1700 loss=0.001605 ss=0.510 ev=event_14 t0=72 n=3703 e=7883
[Model_1] it=1800 loss=0.000008 ss=0.540 ev=event_92 t0=367 n=3698 e=7864
[Model_1] it=1900 loss=0.012056 ss=0.570 ev=event_3 t0=145 n=3698 e=7853
[Model_1] it=2000 loss=0.001924 ss=0.600 ev=event_4 t0=57 n=3692 e=7827
[Model_1] it=2100 loss=0.000668 ss=0.600 ev=event_17 t0=64 n=3695 e=7842
[Model_1] it=2200 loss=0.000016 ss=0.600 ev=event_41 t0=394 n=3702 e=7874
[Model_1] it=2300 loss=0.015896 ss=0.600 ev=event_10 t0=134 n=3704 e=7890
[Model_1] it=2400 loss=0.014874 ss=0.600 ev=event_47 t0=132 n=3695 e=7845
[Model_1] it=2500 loss=0.000711 ss=0.600 ev=event_3 t0=40 n=3700 e=7869
[Model_1] it=2600 loss=0.000010 ss=0.600 ev=event_63 t0=376 n=3702 e=7875
[Model_1] it=2700 loss=0.002034 ss=0.600 ev=event_40 t0=263 n=3692 e=7832
[Model_1] it=2800 loss=0.000367 ss=0.600 ev=event_84 t0=50 n=3696 e=7850
[Model_1] it=2900 loss=0.003519 ss=0.600 ev=event_34 t0=16 n=3695 e=7848
[Model_1] it=3000 loss=0.001578 ss=0.600 ev=event_11 t0=227 n=3698 e=7861
[Model_1] it=3100 loss=0.000007 ss=0.600 ev=event_27 t0=326 n=3692 e=7834
[Model_1] it=3200 loss=0.001429 ss=0.600 ev=event_61 t0=35 n=3698 e=7861
[Model_1] it=3300 loss=0.003420 ss=0.600 ev=event_2 t0=91 n=3692 e=7837
[Model_1] it=3400 loss=0.000005 ss=0.600 ev=event_27 t0=335 n=3701 e=7870
[Model_1] it=3500 loss=0.000050 ss=0.600 ev=event_14 t0=268 n=3699 e=7865
[Model_1] it=3600 loss=0.000045 ss=0.600 ev=event_40 t0=354 n=3702 e=7872
[Model_1] it=3700 loss=0.000555 ss=0.600 ev=event_86 t0=346 n=3699 e=7862
[Model_1] it=3800 loss=0.001728 ss=0.600 ev=event_60 t0=74 n=3695 e=7845
[Model_1] it=3900 loss=0.000005 ss=0.600 ev=event_61 t0=247 n=3701 e=7870
[Model_1] it=4000 loss=0.000004 ss=0.600 ev=event_84 t0=239 n=3695 e=7845
[Model_1] it=4100 loss=0.000006 ss=0.600 ev=event_7 t0=297 n=3695 e=7847
[Model_1] it=4200 loss=0.000451 ss=0.600 ev=event_32 t0=42 n=3703 e=7880
[Model_1] it=4300 loss=0.008328 ss=0.600 ev=event_49 t0=7 n=3703 e=7880
[Model_1] it=4400 loss=0.000949 ss=0.600 ev=event_50 t0=179 n=3697 e=7858
[Model_1] it=4500 loss=0.000392 ss=0.600 ev=event_13 t0=114 n=3691 e=7833
[Model_1] it=4600 loss=0.000006 ss=0.600 ev=event_36 t0=211 n=3702 e=7879
[Model_1] it=4700 loss=0.000012 ss=0.600 ev=event_24 t0=409 n=3701 e=7871
[Model_1] it=4800 loss=0.000699 ss=0.600 ev=event_68 t0=47 n=3699 e=7861
[Model_1] it=4900 loss=0.000003 ss=0.600 ev=event_49 t0=339 n=3700 e=7870
[Model_1] it=5000 loss=0.000003 ss=0.600 ev=event_10 t0=226 n=3707 e=7899
[Model_1] it=5100 loss=0.000002 ss=0.600 ev=event_10 t0=279 n=3701 e=7881
[Model_1] it=5200 loss=0.001418 ss=0.600 ev=event_12 t0=107 n=3699 e=7865
[Model_1] it=5300 loss=0.000002 ss=0.600 ev=event_20 t0=227 n=3696 e=7853
[Model_1] it=5400 loss=0.000817 ss=0.600 ev=event_14 t0=35 n=3697 e=7859
[Model_1] it=5500 loss=0.003425 ss=0.600 ev=event_17 t0=81 n=3697 e=7864
[Model_1] it=5600 loss=0.000008 ss=0.600 ev=event_64 t0=231 n=3706 e=7894
[Model_1] it=5700 loss=0.000009 ss=0.600 ev=event_71 t0=257 n=3697 e=7857
[Model_1] it=5800 loss=0.000396 ss=0.600 ev=event_50 t0=196 n=3701 e=7876
[Model_1] it=5900 loss=0.001503 ss=0.600 ev=event_30 t0=32 n=3700 e=7869
[Model_1] it=6000 loss=0.000001 ss=0.600 ev=event_92 t0=218 n=3705 e=7891
[Model_1] it=6100 loss=0.000342 ss=0.600 ev=event_68 t0=259 n=3702 e=7874
[Model_1] it=6200 loss=0.000003 ss=0.600 ev=event_54 t0=332 n=3700 e=7866
[Model_1] it=6300 loss=5050.991699 ss=0.600 ev=event_19 t0=182 n=3683 e=7803
[Model_1] it=6400 loss=0.001184 ss=0.600 ev=event_40 t0=346 n=3698 e=7863
[Model_1] it=6500 loss=0.006144 ss=0.600 ev=event_43 t0=16 n=3696 e=7854
[Model_1] it=6600 loss=0.000001 ss=0.600 ev=event_23 t0=253 n=3694 e=7844
[Model_1] it=6700 loss=0.000705 ss=0.600 ev=event_11 t0=140 n=3696 e=7854
[Model_1] it=6800 loss=0.000604 ss=0.600 ev=event_30 t0=51 n=3702 e=7872
[Model_1] it=6900 loss=3244.930420 ss=0.600 ev=event_43 t0=173 n=3697 e=7856
[Model_1] it=7000 loss=0.000002 ss=0.600 ev=event_78 t0=316 n=3700 e=7867
[Model_1] it=7100 loss=0.000003 ss=0.600 ev=event_17 t0=387 n=3700 e=7862
[Model_1] it=7200 loss=0.000002 ss=0.600 ev=event_36 t0=208 n=3694 e=7846
[Model_1] it=7300 loss=0.000003 ss=0.600 ev=event_30 t0=374 n=3696 e=7860
[Model_1] it=7400 loss=0.000010 ss=0.600 ev=event_94 t0=389 n=3698 e=7861
[Model_1] it=7500 loss=0.001958 ss=0.600 ev=event_25 t0=234 n=3692 e=7834
[Model_1] it=7600 loss=0.000011 ss=0.600 ev=event_94 t0=293 n=3706 e=7893
[Model_1] it=7700 loss=0.000439 ss=0.600 ev=event_49 t0=91 n=3694 e=7847
[Model_1] it=7800 loss=0.000001 ss=0.600 ev=event_15 t0=285 n=3697 e=7859
[Model_1] it=7900 loss=0.005990 ss=0.600 ev=event_50 t0=119 n=3701 e=7871
[Model_1] it=8000 loss=0.000471 ss=0.600 ev=event_82 t0=158 n=3703 e=7882
[Model_1] it=8100 loss=0.001207 ss=0.600 ev=event_72 t0=32 n=3702 e=7879
[Model_1] it=8200 loss=0.000246 ss=0.600 ev=event_13 t0=263 n=3686 e=7819
[Model_1] it=8300 loss=0.000003 ss=0.600 ev=event_93 t0=182 n=3695 e=7853
[Model_1] it=8400 loss=0.000005 ss=0.600 ev=event_27 t0=365 n=3699 e=7863
[Model_1] it=8500 loss=0.000002 ss=0.600 ev=event_27 t0=337 n=3694 e=7841
[Model_1] it=8600 loss=5023.711914 ss=0.600 ev=event_17 t0=177 n=3704 e=7880
[Model_1] it=8700 loss=0.000001 ss=0.600 ev=event_60 t0=302 n=3697 e=7853
[Model_1] it=8800 loss=0.000002 ss=0.600 ev=event_57 t0=340 n=3697 e=7854
[Model_1] it=8900 loss=0.000001 ss=0.600 ev=event_93 t0=309 n=3702 e=7880
[Model_1] it=9000 loss=0.005242 ss=0.600 ev=event_72 t0=84 n=3697 e=7865
[Model_1] it=9100 loss=0.000310 ss=0.600 ev=event_32 t0=64 n=3696 e=7859
[Model_1] it=9200 loss=0.000001 ss=0.600 ev=event_87 t0=212 n=3699 e=7865
[Model_1] it=9300 loss=0.000352 ss=0.600 ev=event_82 t0=132 n=3696 e=7859
[Model_1] it=9400 loss=0.000002 ss=0.600 ev=event_49 t0=335 n=3706 e=7894
[Model_1] it=9500 loss=0.001060 ss=0.600 ev=event_10 t0=27 n=3695 e=7842
[Model_1] it=9600 loss=0.000001 ss=0.600 ev=event_43 t0=277 n=3709 e=7909
[Model_1] it=9700 loss=0.001103 ss=0.600 ev=event_70 t0=344 n=3692 e=7837
[Model_1] it=9800 loss=0.001161 ss=0.600 ev=event_38 t0=57 n=3698 e=7861
[Model_1] it=9900 loss=0.000001 ss=0.600 ev=event_3 t0=206 n=3702 e=7872
[Model_1] it=10000 loss=0.000001 ss=0.600 ev=event_7 t0=316 n=3697 e=7853
[Model_1] it=10100 loss=0.000024 ss=0.600 ev=event_68 t0=408 n=3702 e=7876
[Model_1] it=10200 loss=0.000038 ss=0.600 ev=event_54 t0=267 n=3691 e=7827
[Model_1] it=10300 loss=0.001020 ss=0.600 ev=event_50 t0=217 n=3701 e=7867
[Model_1] it=10400 loss=0.000008 ss=0.600 ev=event_84 t0=278 n=3702 e=7873
[Model_1] it=10500 loss=0.000262 ss=0.600 ev=event_87 t0=129 n=3697 e=7844
[Model_1] it=10600 loss=0.000002 ss=0.600 ev=event_64 t0=159 n=3693 e=7839
[Model_1] it=10700 loss=0.000004 ss=0.600 ev=event_19 t0=209 n=3701 e=7867
[Model_1] it=10800 loss=0.001141 ss=0.600 ev=event_86 t0=15 n=3698 e=7852
[Model_1] it=10900 loss=0.000019 ss=0.600 ev=event_17 t0=208 n=3698 e=7857
[Model_1] it=11000 loss=0.006578 ss=0.600 ev=event_21 t0=52 n=3700 e=7873
[Model_1] it=11100 loss=5053.207031 ss=0.600 ev=event_45 t0=80 n=3694 e=7843
[Model_1] it=11200 loss=0.000289 ss=0.600 ev=event_20 t0=15 n=3701 e=7873
[Model_1] it=11300 loss=0.000002 ss=0.600 ev=event_54 t0=284 n=3695 e=7846
[Model_1] it=11400 loss=0.000001 ss=0.600 ev=event_24 t0=322 n=3696 e=7858
[Model_1] it=11500 loss=0.000004 ss=0.600 ev=event_14 t0=329 n=3698 e=7854
[Model_1] it=11600 loss=0.000020 ss=0.600 ev=event_78 t0=329 n=3699 e=7867
[Model_1] it=11700 loss=0.002048 ss=0.600 ev=event_93 t0=13 n=3692 e=7835
[Model_1] it=11800 loss=0.000004 ss=0.600 ev=event_93 t0=318 n=3697 e=7859
[Model_1] it=11900 loss=5049.096191 ss=0.600 ev=event_23 t0=190 n=3690 e=7834
[Model_1] it=12000 loss=0.003807 ss=0.600 ev=event_23 t0=95 n=3705 e=7892
[Model_1] it=12100 loss=0.000001 ss=0.600 ev=event_19 t0=260 n=3700 e=7869
[Model_1] it=12200 loss=0.000001 ss=0.600 ev=event_21 t0=270 n=3698 e=7854
[Model_1] it=12300 loss=0.002087 ss=0.600 ev=event_21 t0=111 n=3698 e=7865
[Model_1] it=12400 loss=0.000001 ss=0.600 ev=event_54 t0=207 n=3703 e=7881
[Model_1] it=12500 loss=0.000342 ss=0.600 ev=event_68 t0=340 n=3695 e=7855
[Model_1] it=12600 loss=0.002378 ss=0.600 ev=event_9 t0=27 n=3699 e=7863
[Model_1] it=12700 loss=0.000050 ss=0.600 ev=event_19 t0=169 n=3699 e=7873
[Model_1] it=12800 loss=0.000001 ss=0.600 ev=event_24 t0=207 n=3694 e=7850
[Model_1] it=12900 loss=4825.366211 ss=0.600 ev=event_61 t0=64 n=3699 e=7869
[Model_1] it=13000 loss=0.000020 ss=0.600 ev=event_47 t0=233 n=3701 e=7874
[Model_1] it=13100 loss=0.000009 ss=0.600 ev=event_49 t0=390 n=3698 e=7857
[Model_1] it=13200 loss=0.000449 ss=0.600 ev=event_32 t0=225 n=3703 e=7884
[Model_1] it=13300 loss=0.000001 ss=0.600 ev=event_61 t0=104 n=3701 e=7877
[Model_1] it=13400 loss=0.002795 ss=0.600 ev=event_92 t0=131 n=3697 e=7860
[Model_1] it=13500 loss=0.001197 ss=0.600 ev=event_19 t0=25 n=3700 e=7872
[Model_1] it=13600 loss=0.000000 ss=0.600 ev=event_54 t0=342 n=3700 e=7861
[Model_1] it=13700 loss=0.000002 ss=0.600 ev=event_91 t0=324 n=3698 e=7857
[Model_1] it=13800 loss=0.000007 ss=0.600 ev=event_82 t0=290 n=3689 e=7825
[Model_1] it=13900 loss=0.000430 ss=0.600 ev=event_36 t0=1 n=3699 e=7864
[Model_1] it=14000 loss=0.002391 ss=0.600 ev=event_2 t0=150 n=3696 e=7856
[Model_1] it=14100 loss=0.001963 ss=0.600 ev=event_47 t0=97 n=3702 e=7875
[Model_1] it=14200 loss=0.002126 ss=0.600 ev=event_2 t0=123 n=3695 e=7859
[Model_1] it=14300 loss=0.000172 ss=0.600 ev=event_40 t0=215 n=3698 e=7852
[Model_1] it=14400 loss=0.001817 ss=0.600 ev=event_64 t0=13 n=3702 e=7875
[Model_1] it=14500 loss=0.000003 ss=0.600 ev=event_21 t0=328 n=3696 e=7854
[Model_1] it=14600 loss=0.000001 ss=0.600 ev=event_82 t0=363 n=3700 e=7867
[Model_1] it=14700 loss=0.000002 ss=0.600 ev=event_2 t0=341 n=3694 e=7846
[Model_1] it=14800 loss=0.002240 ss=0.600 ev=event_25 t0=26 n=3703 e=7879
[Model_1] it=14900 loss=0.000261 ss=0.600 ev=event_40 t0=257 n=3696 e=7860
[Model_1] it=15000 loss=5050.370117 ss=0.600 ev=event_64 t0=86 n=3688 e=7824
[Model_1] it=15100 loss=0.001554 ss=0.600 ev=event_78 t0=49 n=3697 e=7857
[Model_1] it=15200 loss=0.000002 ss=0.600 ev=event_15 t0=386 n=3701 e=7871
[Model_1] it=15300 loss=0.000010 ss=0.600 ev=event_41 t0=390 n=3694 e=7850
[Model_1] it=15400 loss=0.000505 ss=0.600 ev=event_4 t0=113 n=3690 e=7830
[Model_1] it=15500 loss=0.005753 ss=0.600 ev=event_50 t0=280 n=3700 e=7867
[Model_1] it=15600 loss=0.000003 ss=0.600 ev=event_19 t0=400 n=3699 e=7869
[Model_1] it=15700 loss=0.000001 ss=0.600 ev=event_93 t0=251 n=3704 e=7889
[Model_1] it=15800 loss=0.000001 ss=0.600 ev=event_82 t0=322 n=3686 e=7805
[Model_1] it=15900 loss=0.002851 ss=0.600 ev=event_85 t0=381 n=3706 e=7886

In [12]:
sub[COL_Y] = sub[COL_Y].astype(np.float32)

# Precompute node_id -> index maps for both node types per model
for model_name in MODELS:
    pack = systems[model_name]["pack"]
    pack["id2i_by_type"] = {
        "2d": pack["id2i_2d"],
        "1d": pack["id2i_1d"],
    }
    pack["N_by_type"] = {
        "2d": pack["N2"],
        "1d": pack["N1"],
    }

# Cache full predictions per (model,event,type) as arrays [T, N]
preds = {}

for model_name in MODELS:
    pack = systems[model_name]["pack"]
    model = trained[model_name]

    for ev in systems[model_name]["test_events"]:
        # 2D
        p2 = predict_event_2d_full(model_name, pack, model, ev, split="test", rounds=2)
        preds[(model_name, ev, "2d")] = p2  # (T, N2)

        # 1D: placeholder (copy warmup truth and hold constant after warmup)
        # You can replace this later with a 1D graph model or coupling.
        mm1 = open_mm_1d(model_name, "test", ev, pack["N1"])
        truth1 = np.asarray(mm1[:, :, 0], dtype=np.float32)
        p1 = np.zeros_like(truth1)
        p1[:WARMUP+1] = truth1[:WARMUP+1]
        p1[WARMUP+1:] = truth1[WARMUP]  # constant after warmup
        preds[(model_name, ev, "1d")] = p1  # (T, N1)

In [13]:
sub_idx = sub.index.to_numpy()

for (model_name, ev, ntype), arr in preds.items():
    msk = (sub[COL_MODEL].astype(str) == model_name) & (sub[COL_EVENT].astype(str) == ev) & (sub[COL_TYPE].astype(str) == ntype)
    idxs = sub_idx[msk.to_numpy()]
    if idxs.shape[0] == 0:
        continue

    pack = systems[model_name]["pack"]
    N = pack["N_by_type"][ntype]
    id2i = pack["id2i_by_type"][ntype]

    # group row order is already by timestep, so position -> timestep
    k = np.arange(idxs.shape[0], dtype=np.int64)
    t = (k // N).astype(np.int64)

    node_ids = sub.loc[idxs, COL_NODE].to_numpy(np.int64)
    node_i = np.fromiter((id2i[int(x)] for x in node_ids), dtype=np.int64, count=node_ids.shape[0])

    sub.loc[idxs, COL_Y] = arr[t, node_i]

out_csv = "submission.csv"
sub.to_csv(out_csv, index=False)
print("Wrote:", out_csv)

KeyboardInterrupt: 